In [13]:
import random
import gymnasium as gym
import numpy as np

In [21]:
def mean_reward(env : gym.Env, 
                pi : callable, 
                n_episodes : int = 100, 
                max_steps : int = 200):
    """
    This function computes the mean reward of a given policy `pi` in the environment `env`.
    It does this by running `n_episodes` episodes, each with a maximum of `max_steps` steps,
    and accumulating the rewards obtained in each episode.
    
    Parameters:
    - env: The environment in which the agent operates.
    - pi: A policy function that takes a state as input and returns an action.
    - n_episodes: The number of episodes to run (default is 100).
    - max_steps: The maximum number of steps allowed per episode (default is 200).
    
    Returns:
    - A float representing the mean rewards (sum of rewards per episode divided by number of episodes).
    """
    results = []
    for _ in range(n_episodes):
        state, _ = env.reset()
        done, steps = False, 0
        results.append(0.0)
        while not done and steps < max_steps:
            state, reward, terminated, truncated, _ = env.step(pi(state))
            done = terminated or truncated
            results[-1] += reward
            steps += 1
    print("# of successful episodes:", sum(r > 0 for r in results), "/", n_episodes)
    return np.mean(results)

In [15]:
def policy_evaluation(pi, P, gamma=1.0, theta=1e-10):
    """
    Evaluate a policy given an environment and a full description of the environment's dynamics.
    This repeatedly updates the state-value function using the given policy
    until it converges to a stable value.
    
    Parameters:
    - pi (callable): A policy function that takes a state as input and returns an action.
    - P (dict): The environment's dynamics.
    - gamma (float): Discount factor.
    - theta (float): A small threshold for determining convergence.
    
    Returns:
    - V (np.ndarray): The state-value function for the given policy.
    """
    prev_V = np.zeros(len(P), dtype=np.float64)
    while True:
        V = np.zeros(len(P), dtype=np.float64)
        for s in range(len(P)):
            for prob, next_state, reward, done in P[s][pi(s)]:
                V[s] += prob * (reward + gamma * prev_V[next_state] * (not done))
        if np.max(np.abs(prev_V - V)) < theta:
            break
        prev_V = V.copy()
    return V


In [16]:
def policy_improvement(V, P, gamma=1.0):
    """
    Given a state-value function V, returns a new improved policy.
    This is a greedy policy with respect to the state-value function V 
    (i.e., it selects actions that maximize the expected return based on V).
     
    Arguments:
        V: State-value function.
        P: Environment dynamics.
        gamma: Discount factor.
    Returns:
        new_pi: Improved policy.
                This is a lambda function that takes a state as input and returns an action.
    """
    Q = np.zeros((len(P), len(P[0])), dtype=np.float64)
    for s in range(len(P)):
        for a in range(len(P[s])):
            for prob, next_state, reward, done in P[s][a]:
                Q[s][a] += prob * (reward + gamma * V[next_state] * (not done))
                
    # we are taking the max in the action dimension (dim=1)
    new_pi = lambda s: {s:a for s, a in enumerate(np.argmax(Q, axis=1))}[s]
    return new_pi

In [17]:
def policy_iteration(P, gamma=1.0, theta=1e-10):
    random_actions = np.random.choice(tuple(P[0].keys()), len(P))
    pi = lambda s: {s:a for s, a in enumerate(random_actions)}[s]
    while True:
        old_pi = {s:pi(s) for s in range(len(P))}
        # first we evaluate the policy by getting the state-value function given current pi
        V = policy_evaluation(pi, P, gamma, theta)
        # then we improve the policy
        pi = policy_improvement(V, P, gamma)
        if old_pi == {s:pi(s) for s in range(len(P))}:
            break
    return V, pi

In [37]:
env  = gym.make("FrozenLake-v1", is_slippery=True, map_name="8x8", success_rate=0.15)
env.reset(seed=42)
policy = lambda s: env.action_space.sample()

print("Mean reward:", mean_reward(env, policy))
env.close()

# of successful episodes: 0 / 100
Mean reward: 0.0


In [40]:
# Let's run policy iteration on the FrozenLake environment
V, pi = policy_iteration(env.unwrapped.P, gamma=0.9)
print("Optimal State-Value Function:\n", V)
print("Mean reward of the optimal policy:", mean_reward(env, pi))
env.close()

Optimal State-Value Function:
 [0.00446267 0.00561238 0.00819593 0.01283996 0.02063024 0.03155537
 0.0443277  0.05158231 0.00451084 0.00570714 0.00842896 0.01343667
 0.02216785 0.03529726 0.05335618 0.06506789 0.0042808  0.00531877
 0.0071864  0.         0.02486705 0.04189345 0.07220046 0.09556467
 0.00444296 0.00566176 0.00848181 0.01397653 0.02805813 0.
 0.10167448 0.15104568 0.0030408  0.0033409  0.00369531 0.
 0.04355464 0.0849094  0.14030493 0.24601576 0.00204754 0.
 0.         0.0225974  0.0558422  0.10802304 0.         0.40530369
 0.00158959 0.         0.00388938 0.00916857 0.         0.17779478
 0.         0.67055337 0.00154721 0.00193675 0.00283263 0.
 0.15777568 0.35679991 0.64910516 0.        ]
# of successful episodes: 45 / 100
Mean reward of the optimal policy: 0.45
